In [ ]:
# This notebood is designed to be run on google colab
# It also needs the split file from https://github.com/DLR-MF-DAS/SSL4EO-S12-v1.1/tree/main/splits
# to be uploaded manually in the same folder structure

In [1]:
%pip install zarr==2.18.7

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.3/211.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 78.8 MB/s eta 0:00:00
  Created wheel for asciitree: filename=asciitree-0.3.3-py3-none-any.whl size=5031 sha256=889d4a55a6f64a6458127cdc3496d69a8331389789c99a635532f8b56f3b7e39
  Stored in directory: /root/.cache/pip/wheels/71/c1/da/23077eb3b87d24d6f3852ed1ed1a1ac2d3c885ad6ebd2b4a07
Successfully built asciitree


In [ ]:
import numpy as np
import torch
import copy
from torchvision import transforms as T
from torchsummary import summary
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
import os
from pathlib import Path
import xarray as xr
import zarr # ensure zarr==2.18.0 if using older zip files
from PIL import Image
from collections import OrderedDict
import random
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
tsne = TSNE()
from PIL import ImageFilter


# device is set to cuda if cuda is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import os
import requests
from pathlib import Path
from tqdm.notebook import tqdm

# === User Configuration ===

LIMIT_GB = 5  # Maximum download limit in GB
MODALITY_S2L1C = False
MODALITY_S2L2A = False
MODALITY_S1GRD = False
MODALITY_S2RGB = True

# === Setup ===

LIMIT_BYTES = LIMIT_GB * 1024 ** 3
SPLIT = "train"
BASE_DIR = Path(f"data/ssl4eo-s12/{SPLIT}")
TOTAL_SIZE = 0

# === Function to Download Files for a Given Modality ===

def download_modality(modality):
    global TOTAL_SIZE

    url_base = f"https://datapub.fz-juelich.de/ssl4eo-s12/{SPLIT}/{modality}"
    target_dir = BASE_DIR / modality
    target_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nDownloading modality: {modality} from {url_base}")

    for i in tqdm(range(1, 10001), desc=f"Downloading {modality}"):
        file_id = f"{i:06d}"
        filename = f"ssl4eos12_{SPLIT}_seasonal_data_{file_id}.zarr.zip"
        url = f"{url_base}/{filename}"
        file_path = target_dir / filename

        if file_path.exists():
            file_size = file_path.stat().st_size
        else:
            try:
                response = requests.get(url, stream=True, timeout=10)
                if response.status_code == 200:
                    with open(file_path, "wb") as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            if chunk:
                                f.write(chunk)
                else:
                    print(f"File not found or download failed: {filename}")
                    break
            except requests.RequestException as e:
                print(f"Error downloading {filename}: {e}")
                break

        # Update total size
        TOTAL_SIZE = sum(f.stat().st_size for f in BASE_DIR.rglob('*') if f.is_file())

        print(f"Total downloaded so far: {TOTAL_SIZE / 1024 / 1024:.2f} MB")

        if TOTAL_SIZE >= LIMIT_BYTES:
            print(f"\nReached the download limit of {LIMIT_GB} GB. Stopping.")
            return

# === Main Execution ===

print(f"Starting SSL4EO-S12 download for the '{SPLIT}' split")
print(f"Download limit: {LIMIT_GB} GB")

# Trigger downloads for selected modalities
if MODALITY_S2L1C: download_modality("S2L1C")
if MODALITY_S2L2A: download_modality("S2L2A")
if MODALITY_S1GRD: download_modality("S1GRD")
if MODALITY_S2RGB: download_modality("S2RGB")

print("Download completed or limit reached.")

In [ ]:
class CustomSSL4EODataset(torch.utils.data.Dataset): # Inherit from torch.utils.data.Dataset
    def __init__(self, data_dir, split_file, transform=None, modalities=['S2RGB'], num_batch_samples=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.modalities = modalities
        self.num_batch_samples = num_batch_samples 
        self.num_timestamps_per_zarr_location = 4 
        self.num_locations_per_zarr_file = 64 

        self.all_sample_identifiers = [] # Stores (relative_zarr_path, location_idx, timestamp_idx) tuples

        # 1. Read the base filenames from the split file
        self.base_zarr_filenames = []
        print(f"Reading split file: {split_file}")
        with open(split_file, 'r') as f:
            for line in f:
                self.base_zarr_filenames.append(line.strip())

        # 2. Build the comprehensive list of all individual image samples
        # Each entry will be a tuple: (modality_name, base_filename, location_idx, timestamp_idx)
        self.all_sample_identifiers = []

        print("Building sample identifiers...")
        for base_filename in self.base_zarr_filenames:
            for modality_name in self.modalities: # Iterate through EACH DESIRED MODALITY
                # Construct the expected full relative path for this specific modality's file
                # Example: "S2RGB/ssl4eos12_train_seasonal_data_000000.zarr.zip"
                relative_modality_file_path = Path(modality_name) / base_filename

                # Check if this specific file path actually exists on disk
                full_check_path = self.data_dir / relative_modality_file_path
                if not full_check_path.exists():
                    # This warning is useful if your split file contains filenames that don't exist
                    # for all modalities you've specified (e.g., a file for S2RGB but not S1GRD).
                    # print(f"Warning: File not found for {modality_name}/{base_filename} at {full_check_path}. Skipping.")
                    continue # Skip this combination if the file doesn't exist

                for loc_idx in range(self.num_locations_per_zarr_file):
                    for ts_idx in range(self.num_timestamps_per_zarr_location):
                        self.all_sample_identifiers.append(
                            (modality_name, base_filename, loc_idx, ts_idx)
                        )

        if not self.all_sample_identifiers:
            raise ValueError(f"No sample identifiers found for modalities {self.modalities} "
                             f"based on split file '{split_file}' and data_dir '{data_dir}'. "
                             "Check file paths and modality names.")

        print(f"Initialized CustomSSL4EODataset with {len(self.all_sample_identifiers)} individual image samples.")

    def __len__(self):
        return len(self.all_sample_identifiers)

    def __getitem__(self, idx):
        # Retrieve the specific identifier tuple for this 'idx'
        modality, base_filename, location_in_file_idx, timestamp_in_sample_idx = self.all_sample_identifiers[idx]

        # Construct the full path to the .zarr.zip file for this specific modality and base filename
        # This is where the modality directory is re-inserted.
        full_zarr_path = self.data_dir / modality / base_filename


        try:
            # Open the Zarr archive using xarray
            ds = xr.open_zarr(full_zarr_path)

            # Access the specific data array (e.g., 'S2RGB' or 'S1GRD')
            image_data_array = ds.bands

            # Select ONE single image slice from the multi-dimensional Zarr DataArray
            # This is critical for getting a (C, H, W) numpy array.
            # Make sure 'location' and 'time' are the correct dimension names in your Zarr.
            single_image_xarray_data = image_data_array.isel({
                'sample': location_in_file_idx,
                'time': timestamp_in_sample_idx
            })

            raw_data_np = single_image_xarray_data.values # Get the numpy array (C, H, W)

            # --- Convert NumPy array (C, H, W) to PIL Image (H, W, C) for torchvision.transforms ---
            if raw_data_np.ndim == 3 and raw_data_np.shape[0] in [1, 3]: # Assuming channels are first (C, H, W)
                 img_array_hwc = np.transpose(raw_data_np, (1, 2, 0)) # Transpose to (H, W, C)
            elif raw_data_np.ndim == 2: # Handle single band images, replicate to 3 channels for MoCo
                img_array_hwc = np.stack([raw_data_np]*3, axis=-1) # Becomes (H, W, 3)
            else:
                
                raise ValueError(f"Raw data from Zarr is not in expected (C, H, W) or (H, W) format after slicing. Shape: {raw_data_np.shape}")

            # Normalize and convert to uint8 for PIL (assuming typical 0-10000 range for S2 data)
            MAX_VAL_FOR_S2 = 10000.0 # Adjust this max value if your data has a different range!
            img_array_normalized_uint8 = (np.clip(img_array_hwc / MAX_VAL_FOR_S2, 0, 1) * 255).astype(np.uint8)

            # Create the PIL Image
            img_pil = Image.fromarray(img_array_normalized_uint8)
# --- Apply MoCoTransform ---
            if self.transform:
                # self.transform (MoCoTransform) expects a PIL Imageprint()
                img_q, img_k = self.transform(img_pil)
            else:
                # Fallback: just convert to tensor if no MoCoTransform is provided
                img_q, img_k = T.ToTensor()(img_pil), T.ToTensor()(img_pil)

            # Return the two augmented tensor views for MoCo
            return {'image1': img_q, 'image2': img_k, 'label': ""}

        except Exception as e:
            print(f"Error in __getitem__ for index {idx}. "
                  f"Attempted to open: {full_zarr_path} "
                  f"(modality: {modality}, location: {location_in_file_idx}, time: {timestamp_in_sample_idx}).")

In [5]:
class GaussianBlur:
    """Gaussian blur augmentation in SimCLR https://arxiv.org/abs/2002.05709"""

    def __init__(self, sigma=[0.1, 2.0]) -> None:
        self.sigma = sigma

    def __call__(self, x):
        sigma = random.uniform(self.sigma[0], self.sigma[1])
        x = x.filter(ImageFilter.GaussianBlur(radius=sigma))
        return x

In [6]:
S2L1C_MEAN = [2607.345, 2393.068, 2320.225, 2373.963, 2562.536, 3110.071, 3392.832, 3321.154, 3583.77, 1838.712, 1021.753, 3205.112, 2545.798]
S2L1C_STD = [786.523, 849.702, 875.318, 1143.578, 1126.248, 1161.98, 1273.505, 1246.79, 1342.755, 576.795, 45.626, 1340.347, 1145.036]

S2L2A_MEAN = [1793.243, 1924.863, 2184.553, 2340.936, 2671.402, 3240.082, 3468.412, 3563.244, 3627.704, 3711.071, 3416.714, 2849.625]
S2L2A_STD = [1160.144, 1201.092, 1219.943, 1397.225, 1400.035, 1373.136, 1429.17, 1485.025, 1447.836, 1652.703, 1471.002, 1365.307]

S1GRD_MEAN = [-12.577, -20.265]
S1GRD_STD = [5.179, 5.872]

S2RGB_MEAN = [100.708, 87.489, 61.932]
S2RGB_STD = [68.550, 47.647, 40.592]


class MoCoTransform:
    def __init__(self, input_size=224, s=1.0):
        # s is the strength of color distortion
        color_jitter = T.ColorJitter(0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s)
        normalize = T.Normalize(mean=S2RGB_MEAN, std=S2RGB_STD)

        self.transform = T.Compose([
            T.RandomResizedCrop(input_size, scale=(0.2, 1.)),
            T.RandomApply([color_jitter], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.RandomApply([GaussianBlur([0.1, 2.0])], p=0.5), # MoCo v2 addition
            T.RandomHorizontalFlip(),
            T.ToTensor(), # Important: Ensure data is converted to Tensor before normalization
            normalize,

        ])

    def __call__(self, x):
        # Returns two augmented views of the same image
        return self.transform(x), self.transform(x)

In [9]:

train_transform = MoCoTransform()
train_dataset = CustomSSL4EODataset(
    data_dir="/content/data/ssl4eo-s12/train",
    split_file="/content/data/ssl4eo-s12/splits/ssl4eos12_train.txt",
    modalities=['S2RGB'], # Specify modality if applicable
    transform=train_transform,
)
BATCHSIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCHSIZE,
    shuffle=True,
    drop_last=True,
    num_workers=2, # Adjust based on your system
    pin_memory=True,
    prefetch_factor=2
)

Reading split file: /content/data/ssl4eo-s12/splits/ssl4eos12_train.txt
Building sample identifiers...
Initialized CustomSSL4EODataset with 32512 individual image samples.


In [ ]:
# dataloaders
dataloader_training_dataset_mutated = train_loader

K = 8192/4

# defining deep learning architecture
resnetq = resnet18(pretrained=None)

classifier = nn.Sequential(OrderedDict([
    ('fc1', nn.Linear(resnetq.fc.in_features, 100)),
    ('added_relu1', nn.ReLU(inplace=True)),
    ('fc2', nn.Linear(100, 50)),
    ('added_relu2', nn.ReLU(inplace=True)),
    ('fc3', nn.Linear(50, 25))
]))

resnetq.fc = classifier
resnetk = copy.deepcopy(resnetq)

# moving the resnet architecture to device
resnetq.to(device)
resnetk.to(device)

τ = 0.05

def loss_function(q, k, queue):

    N = q.shape[0]
    C = q.shape[1]

    pos = torch.exp(torch.div(torch.bmm(q.view(N,1,C), k.view(N,C,1)).view(N, 1),τ))
    neg = torch.sum(torch.exp(torch.div(torch.mm(q.view(N,C), torch.t(queue)),τ)), dim=1)
    denominator = neg + pos

    return torch.mean(-torch.log(torch.div(pos,denominator)))

# Defining data structures for storing training info

losses_train = []

flag = 0

queue = None

# using SGD optimizer
optimizer = optim.SGD(resnetq.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-6)


if not os.path.exists('results'):
    os.makedirs('results')

# load pretrained model, optimizer and training losses file if model.pth file is available
if(os.path.isfile("results/modelq.pth")):
    resnetq.load_state_dict(torch.load("results/modelq.pth"))
    resnetk.load_state_dict(torch.load("results/modelk.pth"))
    optimizer.load_state_dict(torch.load("results/optimizer.pth"))

    # for param_group in optimizer.param_groups:
    #     param_group['weight_decay'] = 1e-6
    #     param_group['lr'] = 0.00001

    temp = np.load("results/lossesfile.npz")
    losses_train = list(temp['arr_0'])
    queue = torch.load("results/queue.pt")


if queue is None:
    while True:

        with torch.no_grad():
            for (_, sample_batched) in enumerate(dataloader_training_dataset_mutated):

                xk = sample_batched['image2']
                xk = xk.to(device)
                k = resnetk(xk)
                k = k.detach()

                k = torch.div(k,torch.norm(k,dim=1).reshape(-1,1))

                if queue is None:
                    queue = k
                else:
                    if queue.shape[0] < K:
                        queue = torch.cat((queue, k), 0)
                    else:
                        flag = 1

                if flag == 1:
                    break

        if flag == 1:
            break


# Boolean variable on whether to perform training or not



/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [ ]:
TRAINING = True
momentum = 0.999
num_epochs = 20



def get_mean_of_list(L):
    return sum(L) / len(L)

if TRAINING:
    # get resnet in train mode
    resnetq.train()

    # run a for loop for num_epochs
    for epoch in range(num_epochs):

        print(epoch)

        # a list to store losses for each epoch
        epoch_losses_train = []

        # run a for loop for each batch
        for (_, sample_batched) in enumerate(dataloader_training_dataset_mutated):

            # zero out grads
            optimizer.zero_grad()

            # retrieve xq and xk the two image batches
            xq = sample_batched['image1']
            xk = sample_batched['image2']

            # move them to the device
            xq = xq.to(device)
            xk = xk.to(device)

            # get their outputs
            q = resnetq(xq)
            k = resnetk(xk)
            k = k.detach()

            q = torch.div(q,torch.norm(q,dim=1).reshape(-1,1))
            k = torch.div(k,torch.norm(k,dim=1).reshape(-1,1))

            # get loss value
            loss = loss_function(q, k, queue)

            # put that loss value in the epoch losses list
            epoch_losses_train.append(loss.cpu().data.item())

            # perform backprop on loss value to get gradient values
            loss.backward()

            # run the optimizer
            optimizer.step()

            # update the queue
            queue = torch.cat((queue, k), 0)

            if queue.shape[0] > K:
                queue = queue[BATCHSIZE:,:]

            # update resnetk
            for θ_k, θ_q in zip(resnetk.parameters(), resnetq.parameters()):
                θ_k.data.copy_(momentum*θ_k.data + θ_q.data*(1.0 - momentum))

        # append mean of epoch losses to losses_train, essentially this will reflect mean batch loss
        losses_train.append(get_mean_of_list(epoch_losses_train))



        # Plot the training losses Graph and save it
        fig = plt.figure(figsize=(10, 10))
        sns.set_style('darkgrid')
        plt.plot(losses_train)
        plt.legend(['Training Losses'])
        plt.savefig('/content/plot/losses.png')
        plt.close()

        # Store model and optimizer files
        # torch.save(resnetq.state_dict(), 'results/modelq.pth')
        # torch.save(resnetk.state_dict(), 'results/modelk.pth')
        # torch.save(optimizer.state_dict(), 'results/optimizer.pth')
        # np.savez("results/lossesfile", np.array(losses_train))
        # torch.save(queue, 'results/queue.pt')


0
1
2
3
4
5
6


In [ ]:
import torchvision.datasets as datasets
from torch.utils.data import DataLoader, random_split

# --- EuroSAT Dataset Setup ---
# EuroSAT images are 64x64. ResNet backbones usually expect 224x224.

# Transforms for training the linear classifier (includes augmentation)
eurosat_transform_train = T.Compose([
    T.RandomResizedCrop(224), # Resize to ResNet expected input size, adds augmentation
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) 
])

# Transforms for validation/testing the linear classifier (no strong augmentation)
eurosat_transform_val = T.Compose([
    T.Resize(256),          # Resize longer side to 256
    T.CenterCrop(224),      # Crop 224x224 from the center
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define the root directory to store EuroSAT data
eurosat_data_root = './eurosat_data'

print(f"Loading FULL EuroSAT dataset from {eurosat_data_root}...")
# Load the full EuroSAT dataset first
full_dataset_eurosat = datasets.EuroSAT(root=eurosat_data_root, download=True)

print(f"Full EuroSAT dataset size: {len(full_dataset_eurosat)} samples.")
print(f"EuroSAT classes: {full_dataset_eurosat.classes}")
num_eurosat_classes = len(full_dataset_eurosat.classes) # Dynamically get number of classes


# Create a random train/validation split
train_size = int(0.8 * len(full_dataset_eurosat)) # 80% for training
val_size = len(full_dataset_eurosat) - train_size # Remaining for validation

train_dataset_eurosat_raw, val_dataset_eurosat_raw = random_split(full_dataset_eurosat, [train_size, val_size])

print(f"EuroSAT training samples (raw): {len(train_dataset_eurosat_raw)}")
print(f"EuroSAT validation samples (raw): {len(val_dataset_eurosat_raw)}")

# Apply the appropriate transforms to the subsets
# This uses a custom Dataset wrapper to apply transforms after splitting
# because random_split returns Subset objects which don't have a 'transform' attribute.
class TransformedSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

train_dataset_eurosat = TransformedSubset(train_dataset_eurosat_raw, transform=eurosat_transform_train)
val_dataset_eurosat = TransformedSubset(val_dataset_eurosat_raw, transform=eurosat_transform_val)

# Create DataLoaders for EuroSAT
train_loader_eurosat = DataLoader(train_dataset_eurosat, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
val_loader_eurosat = DataLoader(val_dataset_eurosat, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

print(f"EuroSAT training samples (transformed): {len(train_dataset_eurosat)}")
print(f"EuroSAT validation samples (transformed): {len(val_dataset_eurosat)}")

In [ ]:
dataloader_training_dataset = train_loader_eurosat
dataloader_testing_dataset = val_loader_eurosat

In [ ]:
# removing the projection head
if len(nn.Sequential(*list(resnetq.fc.children()))) == 5:
    resnetq.fc = nn.Sequential(*list(resnetq.fc.children())[:-3])


Classes_List = full_dataset_eurosat.classes
Classes_Map = dict()
for i in range(len(Classes_List)):
    Classes_Map[Classes_List[i]] = i

# Boolean variable to control whether to train the linear classifier or not
LINEAR = True

class LinearNet(nn.Module):

    def __init__(self):
        super(LinearNet, self).__init__()
        self.fc1 = torch.nn.Linear(100, 10)

    def forward(self, x):
        x = self.fc1(x)
        return(x)

if LINEAR:

    if not os.path.exists('linear'):
        os.makedirs('linear')

    # getting our linear classifier
    linear_classifier = LinearNet()

    # moving it to device
    linear_classifier.to(device)

    # using SGD as a linear optimizer
    linear_optimizer = optim.SGD(linear_classifier.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-6)

    #number of epochs
    num_epochs_linear = 20

    # Boolean variable to control training of linear classifier
    LINEAR_TRAINING = True

    # Defining data structures to store train and test info for linear classifier
    losses_train_linear = []
    acc_train_linear = []
    losses_test_linear = []
    acc_test_linear = []

    # a variable to keep track of the maximum test accuracy, will be useful to store
    # model parameters with the best test accuracy
    max_test_acc = 0

    # if a model exists in the linear folder, load it
    if(os.path.isfile("linear/model.pth")):

        # load state dict for linear model and optimizer
        linear_classifier.load_state_dict(torch.load("linear/model.pth"))
        linear_optimizer.load_state_dict(torch.load("linear/optimizer.pth"))

        # load data structures
        temp = np.load("linear/linear_losses_train_file.npz")
        losses_train_linear = list(temp['arr_0'])
        temp = np.load("linear/linear_losses_test_file.npz")
        losses_test_linear = list(temp['arr_0'])
        temp = np.load("linear/linear_acc_train_file.npz")
        acc_train_linear = list(temp['arr_0'])
        temp = np.load("linear/linear_acc_test_file.npz")
        acc_test_linear = list(temp['arr_0'])

    # Run a for loop for training the linear classifier
    for epoch in range(num_epochs_linear):

        if LINEAR_TRAINING:

            # run linear classifier in train mode
            linear_classifier.train()

            # a list to store losses for each batch in an epoch
            epoch_losses_train_linear = []
            epoch_acc_train_num_linear = 0.0
            epoch_acc_train_den_linear = 0.0

            # for loop for running through each batch
            for (_,  (images, labels)) in enumerate(dataloader_training_dataset):

                # get x and y from the batch
                x = images
                y_actual = labels

                # move them to the device
                x = x.to(device)
                y_actual  = y_actual.to(device)

                with torch.no_grad():
                    # get output from resnet architecture
                    y_intermediate = resnetq(x)

                # zero the grad values
                linear_optimizer.zero_grad()

                # run y_intermediate through the linear classifier
                y_predicted = linear_classifier(y_intermediate)

                # get the cross entropy loss value
                loss = nn.CrossEntropyLoss()(y_predicted, y_actual)

                # add the obtained loss value to this list
                epoch_losses_train_linear.append(loss.data.item())

                # perform backprop through the loss value
                loss.backward()

                # call the linear_optimizer step function
                linear_optimizer.step()

                # get predictions and actual values to cpu
                pred = np.argmax(y_predicted.cpu().data, axis=1)
                actual = y_actual.cpu().data

                #update the numerators and denominators of accuracy
                epoch_acc_train_num_linear += (actual == pred).sum().item()
                epoch_acc_train_den_linear += len(actual)

                x = None
                y_intermediate = None
                y_predicted = None
                sample_batched = None

            # update losses and acc lists
            losses_train_linear.append(get_mean_of_list(epoch_losses_train_linear))
            acc_train_linear.append(epoch_acc_train_num_linear / epoch_acc_train_den_linear)

        # run linear classifier in eval mode
        linear_classifier.eval()

        # essential variables to keep track of losses and acc
        epoch_losses_test_linear = []
        epoch_acc_test_num_linear = 0.0
        epoch_acc_test_den_linear = 0.0

        # run a for loop through each batch
        for (_, (images, labels)) in enumerate(dataloader_testing_dataset):
            x = images
            y_actual = labels
            #y_actual = torch.tensor([Classes_Map[i] for i in y_actual])

            x = x.to(device)
            y_actual  = y_actual.to(device)

            with torch.no_grad():
                y_intermediate = resnetq(x)

            y_predicted = linear_classifier(y_intermediate)
            loss = nn.CrossEntropyLoss()(y_predicted, y_actual)
            epoch_losses_test_linear.append(loss.data.item())

            pred = np.argmax(y_predicted.cpu().data, axis=1)
            actual = y_actual.cpu().data
            epoch_acc_test_num_linear += (actual == pred).sum().item()
            epoch_acc_test_den_linear += len(actual)

        # calculate test_acc
        test_acc = epoch_acc_test_num_linear / epoch_acc_test_den_linear
        print(test_acc)

        if LINEAR_TRAINING:
            losses_test_linear.append(get_mean_of_list(epoch_losses_test_linear))
            acc_test_linear.append(epoch_acc_test_num_linear / epoch_acc_test_den_linear)

            # plotting losses and accuracies

            fig = plt.figure(figsize=(10, 10))
            sns.set_style('darkgrid')
            plt.plot(losses_train_linear)
            plt.plot(losses_test_linear)
            plt.legend(['Training Losses', 'Testing Losses'])
            plt.savefig('/content/linear/losses.png')
            plt.close()

            fig = plt.figure(figsize=(10, 10))
            sns.set_style('darkgrid')
            plt.plot(acc_train_linear)
            plt.plot(acc_test_linear)
            plt.legend(['Training Accuracy', 'Testing Accuracy'])
            plt.savefig('/content/linear/accuracy.png')
            plt.close()

            print("Epoch completed")

            if test_acc >= max_test_acc:

                # save the model only when test_acc exceeds the current max_test_acc

                max_test_acc = test_acc
                torch.save(linear_classifier.state_dict(), 'linear/model.pth')
                torch.save(linear_optimizer.state_dict(), 'linear/optimizer.pth')

        # save data structures
        np.savez("linear/linear_losses_train_file", np.array(losses_train_linear))
        np.savez("linear/linear_losses_test_file", np.array(losses_test_linear))
        np.savez("linear/linear_acc_train_file", np.array(acc_train_linear))
        np.savez("linear/linear_acc_test_file", np.array(acc_test_linear))